## Eddibot - the chatbot that recognizes 5 emotions
For my capstone project for Codecademy's career path, Data Scientist: Natural Language Processing Specialist. Eddibot is a hybrid chatbot that combines a machine-learning emotion classifier with rule-based conversational responses. 
For this project, I used Kaggle's Emotion Detection Text Dataset (https://www.kaggle.com/datasets/abhrajaiswal/emotions-detection-text-dataset). I decided to use this dataset rather than the one provided by Codecademy because I wanted a challenge.
In the next steps, you will be able to read my code and the description of it.

In [5]:
import re
import string
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression

In [19]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\nikaf\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nikaf\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

After brainstorming my project, I decided on this. The first stage of the project was importing the required Python libraries. Pandas and NumPy were used for working with the dataset, while re and string were used during text preprocessing. Scikit-learn provided the tools needed to split the dataset, convert text into numerical features, train classification algorithms, and evaluate their performance. The random module was later used by Eddibot to randomly select questions.

In [8]:
df = pd.read_csv("emotions.txt",sep = ';',header = None,names = ['text','emotion'])

In [10]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


I loaded a labelled emotion dataset into a Pandas DataFrame. Each example contains a piece of text and its associated emotion. Because the correct emotions are already provided, the dataset can be used for supervised machine learning.

In [9]:
df0 = df.copy()

I created a copy of the original DataFrame using. This preserved the original data before transformations were applied. In particular, it allowed the original text-based emotion labels to remain available after the working DataFrame had been converted to numerical labels.

In [77]:
unique_emotions = df0['emotion'].unique()
emotion_numbers = {}
for i, emotion in enumerate(unique_emotions):
    emotion_numbers[emotion] = i
number_to_emotion = {v: k for k, v in emotion_numbers.items()}

In [13]:
df

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1
...,...,...
15995,i just had a very brief time in the beanbag an...,0
15996,i am now turning and i feel pathetic that i am...,0
15997,i feel strong and good overall,5
15998,i feel like this was such a rude comment and i...,1


This code identifies all the unique emotions in the original dataset and assigns a numerical value to each emotion. The enumerate() function is used to automatically number the emotions starting from zero, and these values are stored in the emotion_numbers dictionary. A second dictionary, number_to_emotion, reverses this mapping so that the numerical predictions produced by the machine-learning model can later be converted back into readable emotion names.

In [78]:
def preprocess_text(text):
    # lowercase
    text = text.lower()
    # remove punctuation
    text = remove_punc(text)
    # remove numbers
    text = remove_numbers(text)
    # remove emojis/non-ASCII
    text = remove_emojis(text)
    # remove stopwords
    words = text.split()
    cleaned = []
    for word in words:
        if word not in stop_words:
            cleaned.append(word)
    return " ".join(cleaned)

I created the preprocess_text() function to apply the required cleaning operations to new user input. This ensures that text entered into Eddibot is prepared consistently with the text used to train the machine-learning classifier. I converted the text to lowercase so that words with different capitalisation would be treated as the same word. The remove_punc() function removes punctuation from the text. This reduces unnecessary variation because punctuation does not need to be treated as a separate feature by the TF-IDF vectorizer. The remove_numbers() function removes numerical values from the sentences so that the classifier can focus primarily on words associated with emotions. The remove_emojis() function removes emojis and other characters that were not being used as features by my text-classification approach.I removed common stopwords that occur frequently but may provide relatively little information for distinguishing between emotion classes. This reduces the amount of unnecessary text processed by the classifier.

In [54]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [55]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

I separated the data into input features (X) and target labels (y). The text represents the information given to the model, while the emotion represents the class that the model needs to learn to predict.

In [56]:
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)
pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))

0.768125


I divided the dataset into training and testing sets. The training data was used to teach the models how different words relate to different emotions. The testing data was kept separate so that the trained models could be evaluated using examples they had not been trained on.

In [57]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)
nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


I used TF-IDF, which stands for Term Frequency–Inverse Document Frequency, to transform the text into numerical features. TF-IDF assigns values to words based on their importance within a sentence and across the dataset. This produces numerical data that can be processed by the classification algorithms. I fitted the TF-IDF vectorizer using the training data. The testing data was then transformed using the already fitted vectorizer. This prevents information from the testing data from being used during training. 

The first classification algorithm I tested was Multinomial Naive Bayes. This algorithm is commonly used for text classification tasks because it can classify documents based on the occurrence and importance of different words. 

In [58]:
y_pred = nb2_model.predict(X_test_tfidf)
print(accuracy_score(y_test, y_pred))

0.6609375


The prediction accuracy for the Naive Bayes model was 0.66.

In [59]:
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train_tfidf,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [60]:
log_pred = logistic_model.predict(X_test_tfidf)
print(accuracy_score(y_test,log_pred ))

0.8628125


The second model I tested was Logistic Regression. Although its name contains the word regression, Logistic Regression can be used as a classification algorithm. The model learned the relationship between the TF-IDF features and the emotion classes. Its accuracy is very high - 86%. 

In [81]:
def predict_emotion(text):
    cleaned_text = preprocess_text(text)
    vector = tfidf_vectorizer.transform([cleaned_text])
    prediction = logistic_model.predict(vector)[0]
    emotion = number_to_emotion[int(prediction)]
    return emotion

The predict_emotion() function connects the preprocessing, TF-IDF, and machine-learning stages. It accepts new text, preprocesses it, transforms it into TF-IDF features, and passes those features to the trained Logistic Regression model. The numerical prediction is then converted back into its corresponding emotion name and returned. 

I created the Eddibot class to organise the conversational part of the application. The class contains the possible questions and responses used by the chatbot, as well as methods that control the different stages of the conversation.
Negative_responses stores different ways in which a user may indicate that they do not want to continue with a particular question. This allows Eddibot to recognise more than just the word "no".
Exit_commands contains words that indicate that the user wants to end the conversation. Eddibot checks user responses against these commands so that the conversation can be terminated when requested.
Random_questions stores several questions that Eddibot can ask about the user's emotions. Using multiple questions makes the interaction less repetitive.

### Methods
The greet() method begins the chatbot interaction. It asks for and stores the user's name, introduces Eddibot, and explains the purpose of the chatbot. It then asks whether the user wants to discuss their feelings. If a negative response is detected, the method ends; otherwise, it calls the chat() method.
The make_exit() method determines whether the user wants to end the conversation. It loops through the predefined exit commands and checks whether one appears in the user's response. If an exit command is found, the method returns True; otherwise, it returns False.
The emotion_response() method selects an appropriate chatbot response based on the emotion predicted by the machine-learning model. Each emotion is associated with a predefined response in a dictionary. The .get() method retrieves the correct response, while a default message is available if an unexpected emotion is received. 
The chat() method controls the main interaction between Eddibot and the user. It uses a while loop so that the conversation can continue for multiple interactions rather than ending after a single response.
Random.choice() randomly selects one question from random_questions. This introduces variation into Eddibot's conversation.
After receiving the user's response, Eddibot checks whether it contains an exit command. If make_exit() returns True, break terminates the conversation loop.
The user's response is passed to predict_emotion(). This is the point where the chatbot connects with the machine-learning classifier. The returned emotion is stored in the variable emotion.
After identifying the emotion, Eddibot passes the prediction to emotion_response(). This converts the classification result into a conversational response appropriate to that emotion.
Eddibot displays the emotion predicted by the classifier so that the user can see the result produced by the machine-learning component.
Also, Eddibot asks whether the user would like to provide additional information about their feelings. The response is stored in follow_up.
The follow-up response is also checked for exit commands, allowing the user to end the conversation at this stage.
If the user provides a follow-up statement, it is passed through the same predict_emotion() function. This allows Eddibot to perform emotion classification multiple times during a conversation rather than only analysing the first response.
After defining the Eddibot class, I created an instance of the class called eddi_bot. The class acts as the blueprint, while eddi_bot is the actual chatbot object that can be used by the program.
I called the greet() method on the eddi_bot object to start the program. This begins the interaction and subsequently calls the other methods as required.

In [82]:
class Eddibot:
    negative_responses = ("no", "nope", "nah", "naw", "not a chance", "sorry")
    exit_commands = ("quit", "pause", "exit", "goodbye", "bye", "later")
    random_questions = ("How are you feeling today?", "How have you been lately?", "How would you describe your feelings right now?", "Tell me about how you're feeling.")
    
    def greet(self):
        self.name = input("Hello! What is your name? ")
        print(f"Hi {self.name}, I'm Eddi. "
            "I like talking about emotions such as anger, joy, sadness, love, fear, and surprise.")
        will_talk = input("Would you like to talk about how you're feeling? ").lower()
        if will_talk in self.negative_responses:
            print(f"That's okay, {self.name}. Have a nice day!")
            return
        self.chat()

    def make_exit(self, reply):
        for command in self.exit_commands:
            if command in reply.lower():
                print(f"It was nice talking with you, {self.name}. Goodbye!")
                return True
        return False

    def emotion_response(self, emotion):
        responses = {
            "sadness":
                "It seems to me that you are feeling sad.",

            "joy":
                "That sounds great! I'm sensing joy.",

            "anger":
                "It seems there is some anger inside you.",

            "love":
                "You seem to feel love or affection in this case.",

            "fear":
                "You sound worried or fearful.",

            "surprise":
                "That sounds like a surprise!"
        }
        return responses.get(emotion, f"I think the emotion you're expressing is {emotion}.")

    def chat(self):
        print("\nYou can type 'bye' or 'quit' whenever you want to stop.\n")
        while True:
            question = random.choice(self.random_questions)
            reply = input(f"Eddi: {question}\nYou: ")
            if self.make_exit(reply):
                break
            # Predict the user's emotion
            emotion = predict_emotion(reply)
            print(f"\nEddi: {self.emotion_response(emotion)}")
            print(f"Eddi: My emotion classifier predicted: {emotion}.\n")
            follow_up = input("Eddi: Would you like to tell me more about that?\nYou: ")
            if self.make_exit(follow_up):
                break
            if follow_up.lower() in self.negative_responses:
                print("Eddi: That's okay. We can talk about something else.\n")
                continue
            # Predict emotion of the follow-up too
            follow_up_emotion = predict_emotion(follow_up)
            print(
                f"Eddi: I think that response expresses "
                f"{follow_up_emotion}.\n")
eddi_bot = Eddibot()
eddi_bot.greet()

Hello! What is your name?  Nika


Hi Nika, I'm Eddi. I like talking about emotions such as anger, joy, sadness, love, fear, and surprise.


Would you like to talk about how you're feeling?  yes



You can type 'bye' or 'quit' whenever you want to stop.



Eddi: How are you feeling today?
You:  I'm feeling down.



Eddi: It seems to me that you are feeling sad.
Eddi: My emotion classifier predicted: sadness.



Eddi: Would you like to tell me more about that?
You:  I'm also feeling irritated.


Eddi: I think that response expresses anger.



Eddi: How are you feeling today?
You:  exit


It was nice talking with you, Nika. Goodbye!


My project consists of two connected components: a machine-learning emotion classifier and a rule-based chatbot called Eddibot. I first loaded and preprocessed a labelled emotion dataset. The emotion categories were converted into numerical labels, and TF-IDF was used to transform the text into numerical features. I trained and compared two classification algorithms and selected Logistic Regression because it achieved the highest accuracy in my experiment.

I then created the predict_emotion() function to apply the trained classifier to new text. The function preprocesses the user's sentence, transforms it using the fitted TF-IDF vectorizer, obtains a prediction from the Logistic Regression model, and converts the numerical prediction back into an emotion name.

Finally, I integrated this function into the Eddibot class. Greet() starts the interaction, make_exit() detects when the user wants to leave, chat() controls the conversation, and emotion_response() selects an appropriate predefined response based on the predicted emotion. Therefore, Eddibot is a hybrid chatbot: machine learning is used for emotion classification, while rule-based logic and predefined responses are used to manage the conversation.